# Erro Econômico

In [7]:
import pandas as pd
import numpy as np

def calculate_sku_economic_loss(
    df_eval: pd.DataFrame, 
    id_col: str = "unique_id",
    time_col: str = "ds",
    target_col: str = "y",
    forecast_col: str = "LGBMRegressor",
    cost_understock_col: str = "cu",
    cost_overstock_col: str = "co"
) -> pd.DataFrame:
    """
    Calculates the economic error (financial loss in currency) per SKU and per cycle 
    based on the Newsvendor cost model, directly comparing actual demand against 
    a forecast column (point forecast or quantile), aligned with Nixtla's long-format.

    Parameters
    ----------
    df_eval : pd.DataFrame
        DataFrame containing at least the unique identifier, timestamp, actual demand,
        forecast column, and unit costs.
    id_col : str, default="unique_id"
        Column name for the SKU identifier.
    time_col : str, default="ds"
        Column name for the timestamp.
    target_col : str, default="y"
        Column name for the actual observed demand (y).
    forecast_col : str, default="LGBMRegressor"
        Column name representing the model's prediction (acting as the implied inventory/decision).
    cost_understock_col : str, default="cu"
        Column name for the unit cost of stockout.
    cost_overstock_col : str, default="co"
        Column name for the unit cost of excess.

    Returns
    -------
    df_loss_summary : pd.DataFrame
        A summary DataFrame aggregating the total financial loss per SKU over the evaluation period.
    """
    df = df_eval.copy()
    
    # Se a previsão foi menor que a demanda real, faltou produto (Ruptura / Understock)
    df["understock_units"] = np.maximum(0, df[target_col] - df[forecast_col])
    
    # Se a previsão foi maior que a demanda real, sobrou produto (Excesso / Overstock)
    df["overstock_units"] = np.maximum(0, df[forecast_col] - df[target_col])
    
    # Convertendo unidades em Prejuízo Financeiro
    df["economic_loss"] = (
        (df["understock_units"] * df[cost_understock_col]) + 
        (df["overstock_units"] * df[cost_overstock_col])
    )
    
    # Agregando por SKU
    df_loss_summary = df.groupby(id_col).agg(
        total_economic_loss=("economic_loss", "sum"),
        mean_cycle_loss=("economic_loss", "mean"),
        total_understock_units=("understock_units", "sum"),
        total_overstock_units=("overstock_units", "sum"),
        total_actual_demand=(target_col, "sum")
    ).reset_index()
    
    return df_loss_summary